In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
BRONZE = ROOT / "data" / "bronze"

fs = pd.read_parquet(BRONZE / "fusionsolar.parquet")
so = pd.read_parquet(BRONZE / "solis.parquet")
print(fs.shape, so.shape)

(725604, 10) (509342, 9)


In [3]:
FS_NAMES = {"device_id": "device_hash_id", "collect_time": "ts",
            "active_power": "ac_power_kw", "dc_power": "dc_power_kw",
            "efficiency": "efficiency_pct", "day_cap": "energy_today_kwh",
            "total_cap": "energy_total_kwh", "temperature": "temperature_c",
            "inverter_state": "state_code"}

SO_NAMES = {"inverter_id": "device_hash_id", "timestamp": "ts",
            "total_ac_power": "ac_power_kw", "dc_power": "dc_power_kw",
            "eToday": "energy_today_kwh", "eTotal": "energy_total_kwh",
            "temperature": "temperature_c", "state": "state_code"}

fs = fs.rename(columns=FS_NAMES)
so = so.rename(columns=SO_NAMES)
print(fs.columns.tolist())
print(so.columns.tolist())

['device_hash_id', 'ts', 'ac_power_kw', 'dc_power_kw', 'efficiency_pct', 'energy_today_kwh', 'energy_total_kwh', 'temperature_c', 'state_code', 'source_file']
['device_hash_id', 'ts', 'ac_power_kw', 'dc_power_kw', 'energy_today_kwh', 'energy_total_kwh', 'temperature_c', 'state_code', 'source_file']


In [4]:
NUM_COLS = ["ac_power_kw", "dc_power_kw", "energy_today_kwh",
            "energy_total_kwh", "temperature_c"]

for col in NUM_COLS:
    as_number = pd.to_numeric(so[col], errors="coerce")
    bad = so[col][as_number.isna()]
    print(col, len(bad))
    print(bad.value_counts().head(), "\n")

ac_power_kw 24914
ac_power_kw
0,0     2678
0,08    1409
0,04    1064
0,02     818
NULL     193
Name: count, dtype: int64 

dc_power_kw 24907
dc_power_kw
0,0       2914
NULL       205
N/A        104
0,0161      56
0,0162      42
Name: count, dtype: int64 

energy_today_kwh 24883
energy_today_kwh
0,0     1424
0,1      591
0,2      450
0,3      273
NULL     172
Name: count, dtype: int64 

energy_total_kwh 24869
energy_total_kwh
76445,0    182
77189,0    182
76267,0    182
75894,0    182
76575,0    181
Name: count, dtype: int64 

temperature_c 24894
temperature_c
NULL    200
36,0    179
34,4    161
40,7    160
34,0    155
Name: count, dtype: int64 



In [5]:
MISSING = ["", "NULL", "N/A"]

def to_number(text):
    text = text.str.replace(",", ".", regex=False)  # 32,6 becomes 32.6
    text = text.mask(text.isin(MISSING))            # NULL, N/A, empty become missing
    return pd.to_numeric(text)                      # anything else unexpected = error

for col in NUM_COLS:
    so[col] = to_number(so[col])

print(so[NUM_COLS].dtypes)
print(so[NUM_COLS].isna().sum())

ac_power_kw         float64
dc_power_kw         float64
energy_today_kwh    float64
energy_total_kwh    float64
temperature_c       float64
dtype: object
ac_power_kw         306
dc_power_kw         309
energy_today_kwh    280
energy_total_kwh    277
temperature_c       289
dtype: int64


In [6]:
FS_NUM_COLS = NUM_COLS + ["efficiency_pct"]

for col in FS_NUM_COLS:
    fs[col] = to_number(fs[col])

print(fs[FS_NUM_COLS].dtypes)
print(fs[FS_NUM_COLS].isna().sum())

ac_power_kw         float64
dc_power_kw         float64
energy_today_kwh    float64
energy_total_kwh    float64
temperature_c       float64
efficiency_pct      float64
dtype: object
ac_power_kw         354
dc_power_kw         363
energy_today_kwh    322
energy_total_kwh    345
temperature_c       355
efficiency_pct      344
dtype: int64


In [7]:
print(fs["ts"].head(3))
print(so["ts"].head(3))

0    1735686300000
1    1735686000000
2    1735686000000
Name: ts, dtype: str
0    2024-06-27 17:05:00
1    2024-06-27 18:25:00
2    2024-06-27 17:20:00
Name: ts, dtype: str


In [8]:
has_slash = so["ts"].str.contains("/")
print(has_slash.sum())
print(so.loc[has_slash, "source_file"].nunique())

23877
17


In [9]:
fs["ts"] = pd.to_numeric(fs["ts"])
fs["ts"] = pd.to_datetime(fs["ts"], unit="ms", utc=True)
print(fs["ts"].head(3))
print(fs["ts"].dtype)

0   2024-12-31 23:05:00+00:00
1   2024-12-31 23:00:00+00:00
2   2024-12-31 23:00:00+00:00
Name: ts, dtype: datetime64[us, UTC]
datetime64[us, UTC]


In [10]:
normal = pd.to_datetime(so["ts"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
dayfirst = pd.to_datetime(so["ts"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
so["ts"] = normal.fillna(dayfirst)
print(so["ts"].isna().sum())
print(so["ts"].head(3))

0
0   2024-06-27 17:05:00
1   2024-06-27 18:25:00
2   2024-06-27 17:20:00
Name: ts, dtype: datetime64[us]


In [11]:
so["ts"] = so["ts"] - pd.Timedelta(hours=1)
so["ts"] = so["ts"].dt.tz_localize("UTC")
print(so["ts"].head(3))
print(so["ts"].dtype, fs["ts"].dtype)

0   2024-06-27 16:05:00+00:00
1   2024-06-27 17:25:00+00:00
2   2024-06-27 16:20:00+00:00
Name: ts, dtype: datetime64[us, UTC]
datetime64[us, UTC] datetime64[us, UTC]


In [12]:
devices = pd.read_parquet(BRONZE / "devices.parquet")
known = devices["device_hash_id"]

fs_bad = fs.loc[~fs["device_hash_id"].isin(known), "device_hash_id"]
so_bad = so.loc[~so["device_hash_id"].isin(known), "device_hash_id"]
print(len(fs_bad), len(so_bad))
print(fs_bad.value_counts().head(5))
print(so_bad.value_counts().head(5))

1663 974
device_hash_id
0000deadbeef0000deadbeef00000000       276
  94aedd81bad035cc73968fe5b06f808a      77
  6187ec08ac3e790c94c9ae81b0af5e1c      51
  dc8fae88167f73e17a152555e7580c1b      48
  22a8a8cda7e139d01f0b204f4a7d80e8      47
Name: count, dtype: int64
device_hash_id
29F08F60676EA3C4E2085071C1957D8C    124
2E1392D8173E175E4FA40A27AA555E09    101
A2BC58707E3AC5B41A4A8F49900D675B     90
372F9D827236B14028B1639E7548A7D2     86
594DF11D8E82584B2EE6278A7CAC9471     76
Name: count, dtype: int64


In [13]:
fs["device_hash_id"] = fs["device_hash_id"].str.strip().str.lower()
so["device_hash_id"] = so["device_hash_id"].str.strip().str.lower()

fs["unknown_device"] = ~fs["device_hash_id"].isin(known)
so["unknown_device"] = ~so["device_hash_id"].isin(known)
print(fs["unknown_device"].sum(), so["unknown_device"].sum())

276 0


In [14]:
print(fs.duplicated(subset=["device_hash_id", "ts"]).sum())
print(so.duplicated(subset=["device_hash_id", "ts"]).sum())

31729
23002


In [15]:
fs = fs.sort_values("source_file").drop_duplicates(subset=["device_hash_id", "ts"], keep="last")
so = so.sort_values("source_file").drop_duplicates(subset=["device_hash_id", "ts"], keep="last")
print(len(fs), len(so))

693875 486340


In [16]:
fs["bad_temperature"] = fs["temperature_c"] > 100
so["bad_temperature"] = so["temperature_c"] > 100

fs["bad_counter"] = fs["energy_total_kwh"] == 0
so["bad_counter"] = so["energy_total_kwh"] == 0

fs_states = pd.read_parquet(BRONZE / "inverter_state_mapping_fusionsolar.parquet")
so_states = pd.read_parquet(BRONZE / "inverter_state_mapping_solis.parquet")
fs["unknown_state"] = ~fs["state_code"].isin(fs_states["state_id"])
so["unknown_state"] = ~so["state_code"].isin(so_states["state_id"])

for col in ["bad_temperature", "bad_counter", "unknown_state"]:
    print(col, fs[col].sum(), so[col].sum())

bad_temperature 136 120
bad_counter 199 139
unknown_state 335 226


In [17]:
devices["max_power"] = pd.to_numeric(devices["max_power"])
size = devices[["device_hash_id", "max_power"]]

fs = fs.merge(size, on="device_hash_id", how="left")
so = so.merge(size, on="device_hash_id", how="left")

fs["too_much_power"] = fs["ac_power_kw"] > fs["max_power"] * 2
so["too_much_power"] = so["ac_power_kw"] > so["max_power"] * 2
print(fs["too_much_power"].sum(), so["too_much_power"].sum())
print(fs.loc[fs["too_much_power"], "source_file"].nunique())

1249 13
10


In [18]:
fs["vendor"] = "fusionsolar"
so["vendor"] = "solis"

silver = pd.concat([fs, so], ignore_index=True)
silver = silver.drop(columns=["max_power"])
print(silver.shape)
print(silver.dtypes)

SILVER = ROOT / "data" / "silver"
SILVER.mkdir(exist_ok=True)
silver.to_parquet(SILVER / "readings.parquet", index=False)

(1180215, 16)
device_hash_id                      str
ts                  datetime64[us, UTC]
ac_power_kw                     float64
dc_power_kw                     float64
efficiency_pct                  float64
energy_today_kwh                float64
energy_total_kwh                float64
temperature_c                   float64
state_code                          str
source_file                         str
unknown_device                     bool
bad_temperature                    bool
bad_counter                        bool
unknown_state                      bool
too_much_power                     bool
vendor                              str
dtype: object


In [19]:
fs_states["vendor"] = "fusionsolar"
so_states["vendor"] = "solis"
states = pd.concat([fs_states, so_states])
states = states[["vendor", "state_id", "canonical_state_id"]]
states = states.rename(columns={"state_id": "state_code"})

silver = silver.merge(states, on=["vendor", "state_code"], how="left")
silver["state"] = silver["canonical_state_id"].map({"1": "ON", "2": "OFF", "3": "ERROR"})

print(len(silver))
print(silver.groupby("vendor")["state"].value_counts(dropna=False))

1180215
vendor       state
fusionsolar  ON       601478
             OFF       89012
             ERROR      3050
             NaN         335
solis        ON       486114
             NaN         226
Name: count, dtype: int64
